# DART 공시 본문 텍스트 수집

## 변경된 접근 방식
- 이전: 공시 **제목(report_nm)** 기반 → 키워드만으로 분류 가능 → BERT 불필요
- 이번: 공시 **본문 텍스트** 기반 → 내용을 이해해야 분류 가능 → BERT 필요

## 수집 흐름
```
공시 목록 API (rcept_no 확보)
        ↓
document.xml API (rcept_no → ZIP 다운로드)
        ↓
HTML 파싱 → 본문 텍스트 추출
        ↓
dart_corpus_text.csv 저장
```

## 카테고리 (3개)
| 카테고리 | 본문 특징 |
|---|---|
| 사업보고서 | 사업 현황, 재무 요약, 리스크 요인 |
| 유상증자 | 발행 목적, 조달 금액, 주주 희석 내용 |
| 감사보고서 | 감사 의견, 핵심 감사 사항, 내부 통제 |

## 0. 라이브러리 설치

## 1. API 키 및 기본 설정

In [ ]:
import os
import requests
import pandas as pd
import zipfile
import io
import time
import re
from tqdm import tqdm
from bs4 import BeautifulSoup

API_KEY = os.environ.get("DART_API_KEY", "")
LIST_URL = "https://opendart.fss.or.kr/api/list.json"
DOC_URL  = "https://opendart.fss.or.kr/api/document.xml"

print("설정 완료")

## 2. 공시 목록 수집 함수
이번엔 `rcept_no`를 반드시 포함한다.

In [ ]:
def collect_list(bgn_de, end_de, pblntf_ty, max_pages=5):
    all_items = []
    
    for page in range(1, max_pages + 1):
        params = {
            "crtfc_key": API_KEY,  
            "bgn_de": bgn_de,
            "end_de": end_de,
            "pblntf_ty": pblntf_ty,
            "page_no": page,
            "page_count": 100
        }
        
        try:
            res = requests.get(LIST_URL, params=params, timeout=10)
            data = res.json()
            
            status = data.get('status')
            message = data.get('message', '메시지 없음')
            
            if status != '000':
                if status != '013':
                    print(f"[{bgn_de}] Status: {status}, Msg: {message}")
                break
                
            items = data.get('list', [])
            if not items:
                break
                
            all_items.extend(items)
            
            tp = data.get('total_page', 1)
            if page >= int(tp if tp else 1):
                break
                
            time.sleep(0.3)
            
        except Exception as e:
            print(f"네트워크 오류: {e}")
            break
            
    return pd.DataFrame(all_items)

print("함수 디버깅 버전 정의 완료")

## 3. 공시 본문 텍스트 추출 함수
DART document.xml API → ZIP → HTML 파싱 → 텍스트

In [ ]:
def fetch_document_text(rcept_no, max_chars=1500):
    try:
        params = {"crtfc_key": API_KEY, "rcept_no": rcept_no}
        res = requests.get(DOC_URL, params=params, timeout=15)
        
        if res.status_code != 200:
            return ""

        with zipfile.ZipFile(io.BytesIO(res.content)) as z:
            candidates = [f for f in z.namelist()
                          if f.endswith('.htm') or f.endswith('.xml')]
            if not candidates:
                return ""
            
            main_file = max(candidates, key=lambda f: z.getinfo(f).file_size)
            
            with z.open(main_file) as f:
                content = f.read()
                
                raw = None
                for encoding in ['euc-kr', 'cp949', 'utf-8']:
                    try:
                        raw = content.decode(encoding)
                        break
                    except (UnicodeDecodeError, TypeError):
                        continue
                
                if raw is None:
                    raw = content.decode('utf-8', errors='ignore')

        soup = BeautifulSoup(raw, 'lxml')
        for tag in soup(['script', 'style', 'table']): 
            tag.decompose()

        text = soup.get_text(separator=' ', strip=True)
        text = re.sub(r'&cr;', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        
        return text[:max_chars]

    except Exception as e:
        return ""
    
print("함수 정의 완료")

## 4. 카테고리별 공시 목록 수집

In [ ]:
PERIODS = [
    ("20220101", "20220331"), ("20220401", "20220630"),
    ("20220701", "20220930"), ("20221001", "20221231"),
    ("20230101", "20230331"), ("20230401", "20230630"),
    ("20230701", "20230930"), ("20231001", "20231231"),
    ("20240101", "20240331"), ("20240401", "20240630"),
    ("20240701", "20240930"), ("20241001", "20241231"),
]

SAMPLE_PER_PERIOD = 30  

print("[1/3] 사업보고서 목록 수집 중...")
biz_list = []
for bgn, end in tqdm(PERIODS):
    df = collect_list(bgn, end, pblntf_ty="A", max_pages=30)
    if not df.empty:
        df = df[df['report_nm'].str.contains('사업보고서', na=False)].head(SAMPLE_PER_PERIOD).copy()
        df['label'] = '사업보고서'
        biz_list.append(df)

print("[2/3] 유상증자 목록 수집 중...")
cap_list = []
for bgn, end in tqdm(PERIODS):
    df = collect_list(bgn, end, pblntf_ty="B", max_pages=30)
    if not df.empty:
        df = df[df['report_nm'].str.contains('유상증자', na=False)].head(SAMPLE_PER_PERIOD).copy()
        df['label'] = '유상증자'
        cap_list.append(df)

print("[3/3] 감사보고서 목록 수집 중...")
audit_list = []
for bgn, end in tqdm(PERIODS):
    df = collect_list(bgn, end, pblntf_ty="F", max_pages=20)
    if not df.empty:
        df = df[df['report_nm'].str.contains('감사보고서', na=False)].head(SAMPLE_PER_PERIOD).copy()
        df['label'] = '감사보고서'
        audit_list.append(df)

meta_df = pd.concat(biz_list + cap_list + audit_list, ignore_index=True)
meta_df = meta_df[['rcept_no', 'corp_name', 'report_nm', 'rcept_dt', 'label']]
meta_df = meta_df.drop_duplicates(subset=['rcept_no'])

print(f"\n목록 수집 완료: {len(meta_df):,}개")
print(meta_df['label'].value_counts())

## 5. 공시 본문 텍스트 수집

In [ ]:
print(f"총 {len(meta_df)}개 공시 본문 수집 시작...")
print(f"예상 시간: 약 {round(len(meta_df) * 1.5 / 60, 1)}분")

texts = []
for rcept_no in tqdm(meta_df['rcept_no']):
    text = fetch_document_text(rcept_no, max_chars=1500)
    texts.append(text)
    time.sleep(0.5)

meta_df = meta_df.copy()
meta_df['text'] = texts

before = len(meta_df)
meta_df = meta_df[meta_df['text'].str.len() > 100]
print(f"\n텍스트 추출 성공: {len(meta_df)}/{before}개")
print(meta_df['label'].value_counts())

In [ ]:
# 텍스트 샘플 확인
for label in meta_df['label'].unique():
    sample = meta_df[meta_df['label'] == label].iloc[0]
    print(f"\n{'='*60}")
    print(f"[{label}] {sample['corp_name']} - {sample['report_nm']}")
    print(f"본문: {sample['text'][:300]}...")

## 6. 저장

In [ ]:
SAVE_PATH = "./data/dart_corpus_text.csv"
meta_df.to_csv(SAVE_PATH, index=False, encoding='utf-8-sig')

print(f"저장 완료: {SAVE_PATH}")
print(f"총 {len(meta_df):,}개 샘플")
print()
print(meta_df['label'].value_counts())
print()
print("텍스트 길이 통계:")
print(meta_df['text'].str.len().describe())